# Belka Gen AI — Notebook 01: avaliação do RAG

**Objetivo:** apresentar e interpretar os resultados **já obtidos** da busca por palavras do RAG, sem executar a Qwen, consultar APIs externas ou repetir avaliações. Os números são lidos diretamente de `avaliacao/metricas_busca_executadas.json`; os casos de teste vêm de `avaliacao/casos_teste.json`.

**Origem:** resultados armazenados no repositório Belka Gen AI. A data registrada pelo arquivo de métricas é um metadado do experimento, não a data de execução deste notebook.

In [14]:
import json
from pathlib import Path

# Funciona ao abrir o notebook na raiz do repositório ou na pasta notebooks/.
candidatos = [Path.cwd(), Path.cwd().parent]
raiz = next((p for p in candidatos if (p / "avaliacao" / "metricas_busca_executadas.json").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra este notebook dentro do projeto Belka: falta avaliacao/metricas_busca_executadas.json")

metricas = json.loads((raiz / "avaliacao" / "metricas_busca_executadas.json").read_text(encoding="utf-8"))
casos = json.loads((raiz / "avaliacao" / "casos_teste.json").read_text(encoding="utf-8"))
assert len(metricas["resultados"]) == 5, "Esperados cinco limiares no arquivo de métricas."
assert len(casos) == 17, "Esperados 17 casos de teste."
print("Arquivo de métricas:", (raiz / "avaliacao" / "metricas_busca_executadas.json").relative_to(raiz))
print("Data registrada no arquivo:", metricas.get("data_execucao", "não informada"))
print("Fontes:", metricas["fontes"], "| Trechos:", metricas["trechos"])
print("Casos respondíveis:", sum(bool(c["respondivel"]) for c in casos), "| Não consta:", sum(not c["respondivel"] for c in casos))

Arquivo de métricas: avaliacao\metricas_busca_executadas.json
Data registrada no arquivo: 2026-11-09
Fontes: 9 | Trechos: 24
Casos respondíveis: 12 | Não consta: 5


## 1. Recall@1, Recall@3 e falsos positivos

**Recall@k:** proporção das perguntas respondíveis cujo ID de trecho esperado apareceu entre os `k` primeiros resultados. **Falso positivo:** pergunta marcada como “não consta” para a qual a busca retornou ao menos um trecho. Esses indicadores avaliam recuperação, não garantem que a resposta gerada seja correta.

In [15]:
# Mostra as métricas de busca para cada limite de pontuação.
print(f"{'Limiar':>8} | {'Recall@1':>9} | {'Recall@3':>9} | {'Falsos positivos':>16}")
print("-" * 53)
for r in metricas["resultados"]:
    print(f"{r['limiar']:8.2f} | {r['recall_at_1']:8.1%} | {r['recall_at_3']:8.1%} | {r['falsos_positivos_nao_consta']:16d}")

  Limiar |  Recall@1 |  Recall@3 | Falsos positivos
-----------------------------------------------------
    0.20 |    66.7% |   100.0% |                1
    0.50 |    66.7% |   100.0% |                1
    1.00 |    66.7% |   100.0% |                1
    1.50 |    66.7% |    91.7% |                0
    2.00 |    50.0% |    66.7% |                0


## 2. Decisão sobre o limiar

O limiar **0,20** foi mantido no projeto por preservar Recall@3 de **100% (12/12)** nos casos respondíveis. A escolha aceita **um falso positivo em cinco** casos “não consta”. O limiar 1,50 reduziu falsos positivos a zero, mas baixou Recall@3 para aproximadamente 91,7%.

Essa escolha atende ao objetivo de encontrar conhecimento relevante, mas **não significa que um trecho recuperado sempre fundamenta adequadamente uma resposta**. Não houve comprovação de generalização para perguntas ou fontes inéditas.

In [16]:
# Compara o limite escolhido com uma opção mais restritiva.
selecionado = next(r for r in metricas["resultados"] if abs(r["limiar"] - 0.20) < 1e-9)
conservador = next(r for r in metricas["resultados"] if abs(r["limiar"] - 1.50) < 1e-9)
print("Limiar selecionado:", selecionado["limiar"])
print("Recall@1:", f"{selecionado['recall_at_1']:.1%}")
print("Recall@3:", f"{selecionado['recall_at_3']:.1%}")
print("Falsos positivos entre não consta:", f"{selecionado['falsos_positivos_nao_consta']}/{selecionado['casos_nao_consta']}")
print("Alternativa 1,50 — Recall@3:", f"{conservador['recall_at_3']:.1%}", "| falsos positivos:", conservador['falsos_positivos_nao_consta'])

Limiar selecionado: 0.2
Recall@1: 66.7%
Recall@3: 100.0%
Falsos positivos entre não consta: 1/5
Alternativa 1,50 — Recall@3: 91.7% | falsos positivos: 0


## 3. Cobertura dos casos de teste e ressalvas

Os 17 casos incluem 12 perguntas respondíveis e cinco situações classificadas como “não consta”. A avaliação do recuperador compara palavras e utiliza correspondência com IDs de gabarito, o que **não mede por si só** correção do conteúdo. Exemplos de análise qualitativa estão em `docs/analise_cinco_erros.md`, no relatório técnico e nos resultados comparativos.

**Reprodutibilidade:** este notebook usa somente a biblioteca padrão do Python e dois arquivos JSON já registrados no GitHub. Para refazer a avaliação da busca do zero, há o script `scripts/avaliar_busca.py`, mas ele **não é executado aqui**.

In [17]:
# Lista os casos de teste e confere se há IDs repetidos.
respondiveis = [c["id"] for c in casos if c["respondivel"]]
nao_consta = [c["id"] for c in casos if not c["respondivel"]]
print("IDs respondíveis:", ", ".join(respondiveis))
print("IDs não consta:", ", ".join(nao_consta))
print("Integridade dos IDs:", "OK" if len({c["id"] for c in casos}) == len(casos) else "ERRO: IDs duplicados")

IDs respondíveis: T01, T02, T03, T04, T05, T06, T07, T08, T09, T10, T11, T12
IDs não consta: T13, T14, T15, T16, T17
Integridade dos IDs: OK
